# Serve Qwen3.8-27B on Kaggle free tier

Gives your laptop an **OpenAI-compatible endpoint** on a public HTTPS URL.

## Sidebar, once

**Accelerator = GPU T4 x2**, and **Internet = ON**.

## The caching plan, and why it is split

`/kaggle/working` is capped at **20 GB** for anything you save. The model alone is ~19.5 GB, so **model plus wheels will not fit** and cannot be saved as one output. It is therefore split:

| What | Size | How it is cached | Saves |
|---|---|---|---|
| **vllm wheels + cloudflared** | ~2-3 GB | Save Version, fits easily | The slow part: pip dependency resolution |
| **Model weights** | ~19.5 GB | Downloaded each run, accelerated | Nothing, but `hf-xet` makes it fast |

Dependency resolution is the genuinely slow, CPU-bound part. Model download is bandwidth-bound and Kaggle's network is quick.

**If the model download turns out slow in practice**, see the last markdown cell for caching it too as a 200 GB-limit Dataset, which needs an API token.

## How to use it

1. **Once:** `BUILD_CACHE = True`, Run All, then **Save Version** (Save & Run All). That publishes `/kaggle/working` as this notebook's Output.
2. **Every time after:** `BUILD_CACHE = False`, and **Add Input → Notebook Output → this notebook**. Then Run All.

## Stopping

Click **Stop Session** in the sidebar when you finish. Nothing else stops the quota.

In [ ]:
# ============================================================
#  CONFIG - the only cell you edit
# ============================================================

BUILD_CACHE = False    # True once to build the wheel cache. False after.

# Path to the attached notebook output / dataset holding wheels/ and bin/
CACHE = "/kaggle/input/qwen27b-serving-cache"

# Optional: if you later cache the 19.5 GB model as its own Dataset,
# put its path here to skip the download. None = download each run.
MODEL_CACHE = None

RUN_MINUTES = 120      # auto-stop the model after this long
MAX_LEN     = 8192     # lower to 4096 if you hit CUDA out-of-memory
DL_WORKERS  = 16       # parallel file downloads for the model

MODEL_REPO  = "philbert440/Qwen3.8-27B-W4A16-AWQ"   # ~19.5 GB, fits 2x T4
# fallback: "cyankiwi/Qwen3.8-27B-AWQ-INT4"          # ~21 GB, more widely used

import os, time, datetime
T0 = time.time()

def step(msg):
    """Timestamped progress, so a long pause is obviously not a hang."""
    print(f"[{datetime.datetime.now():%H:%M:%S}  +{time.time()-T0:5.0f}s]  {msg}", flush=True)

HAVE_CACHE = (not BUILD_CACHE) and os.path.isdir(f"{CACHE}/wheels")

step("MODE: " + ("BUILD CACHE (one-time)" if BUILD_CACHE
                 else ("serve, wheels cached" if HAVE_CACHE
                       else "serve, NO CACHE - will be slow")))
if not BUILD_CACHE and not HAVE_CACHE:
    step(f"  no wheels at {CACHE}/wheels")
    step("  Add Input -> Notebook Output, or set BUILD_CACHE=True once.")

In [ ]:
# ONE-TIME cache build. Skipped unless BUILD_CACHE is True.
# Output stays well under the 20 GB /kaggle/working limit.

if BUILD_CACHE:
    import subprocess
    OUT = "/kaggle/working"
    os.makedirs(f"{OUT}/wheels", exist_ok=True)
    os.makedirs(f"{OUT}/bin", exist_ok=True)

    step("downloading vllm wheels (verbose, a few minutes)")
    # No -q and no capture: pip streams progress straight into the notebook.
    subprocess.run(["pip", "download", "vllm", "-d", f"{OUT}/wheels",
                    "--progress-bar", "on"], check=True)

    step("downloading cloudflared")
    subprocess.run(["wget", "--progress=bar:force:noscroll",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", f"{OUT}/bin/cloudflared"], check=True)

    n = len(os.listdir(f"{OUT}/wheels"))
    sz = subprocess.run(["du", "-sh", f"{OUT}/wheels", f"{OUT}/bin"],
                        capture_output=True, text=True).stdout.strip()
    step(f"done: {n} wheels")
    print(sz)
    print("\n" + "="*70)
    print("  CACHE BUILT.")
    print("  1. Click 'Save Version' -> 'Save & Run All'. This publishes")
    print("     /kaggle/working as this notebook's Output.")
    print("  2. Set BUILD_CACHE = False.")
    print("  3. Add Input -> Notebook Output -> this notebook.")
    print("  4. Click Stop Session.")
    print("="*70)
    CACHE_DONE = True    # later cells check this and skip
else:
    CACHE_DONE = False

In [ ]:
# Skipped when BUILD_CACHE is True. Jupyter swallows SystemExit, so
# "Run All" would otherwise carry on and rebuild the environment.
if CACHE_DONE:
    raise RuntimeError("BUILD_CACHE run finished. Save Version, set "
                       "BUILD_CACHE=False, attach the output, then rerun.")

# 1. Hardware check, then install. Everything verbose.

import torch, subprocess, shutil

n = torch.cuda.device_count()
assert n >= 2, "Set Accelerator to 'GPU T4 x2' in the sidebar, then Run All again."
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    step(f"GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if HAVE_CACHE:
    step("installing vllm from cached wheels, offline")
    subprocess.run(["pip", "install", "--no-index", "--find-links",
                    f"{CACHE}/wheels", "vllm", "--progress-bar", "on"], check=True)
    shutil.copy(f"{CACHE}/bin/cloudflared", "/usr/local/bin/cloudflared")
else:
    step("installing vllm from the network (slow, no cache)")
    subprocess.run(["pip", "install", "vllm", "--progress-bar", "on"], check=True)
    subprocess.run(["wget", "--progress=bar:force:noscroll",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"], check=True)

os.chmod("/usr/local/bin/cloudflared", 0o755)
step("install complete")

In [ ]:
# Skipped when BUILD_CACHE is True. Jupyter swallows SystemExit, so
# "Run All" would otherwise carry on and rebuild the environment.
if CACHE_DONE:
    raise RuntimeError("BUILD_CACHE run finished. Save Version, set "
                       "BUILD_CACHE=False, attach the output, then rerun.")

# 2. Get the model, with a progress bar and parallel downloads.
#
#    hf-xet is the current fast backend. hf_transfer is deprecated: the Hub
#    moved to Xet, which does chunk-level dedup, so hf_transfer no longer applies.

if MODEL_CACHE and os.path.isdir(MODEL_CACHE):
    MODEL_PATH = MODEL_CACHE
    step(f"using cached model at {MODEL_PATH}")
else:
    step("ensuring hf-xet fast backend is present")
    subprocess.run(["pip", "install", "-U", "hf-xet", "--progress-bar", "on"], check=False)

    from huggingface_hub import snapshot_download
    # /kaggle/tmp has ~60 GB and is not subject to the 20 GB working limit.
    MODEL_PATH = "/kaggle/tmp/model"
    step(f"downloading {MODEL_REPO} (~19.5 GB) with {DL_WORKERS} workers")
    step("progress bars follow; a stalled bar means network, not a hang")
    snapshot_download(repo_id=MODEL_REPO, local_dir=MODEL_PATH,
                      max_workers=DL_WORKERS)

sz = subprocess.run(["du", "-sh", MODEL_PATH], capture_output=True, text=True).stdout.strip()
step(f"model ready: {sz}")

In [ ]:
# Skipped when BUILD_CACHE is True. Jupyter swallows SystemExit, so
# "Run All" would otherwise carry on and rebuild the environment.
if CACHE_DONE:
    raise RuntimeError("BUILD_CACHE run finished. Save Version, set "
                       "BUILD_CACHE=False, attach the output, then rerun.")

# 3. Start the model server.
#    T4 is Turing: no bfloat16, no FlashAttention-2, no Marlin kernels.

import secrets

PORT, API_KEY, LOG = 8000, secrets.token_urlsafe(24), "/kaggle/working/vllm.log"

server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_PATH,
    "--served-model-name", "qwen",
    "--api-key", API_KEY,
    "--port", str(PORT),
    "--tensor-parallel-size", "2",
    "--dtype", "float16",
    "--quantization", "awq",
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.92",
    "--enforce-eager",
    "--limit-mm-per-prompt", "image=0,video=0",
], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

step(f"vLLM starting, pid {server.pid} - loading weights takes a few minutes")

In [ ]:
# Skipped when BUILD_CACHE is True. Jupyter swallows SystemExit, so
# "Run All" would otherwise carry on and rebuild the environment.
if CACHE_DONE:
    raise RuntimeError("BUILD_CACHE run finished. Save Version, set "
                       "BUILD_CACHE=False, attach the output, then rerun.")

# 4. Wait for load, streaming the log so you can see it is alive.

import requests, re

def die(msg):
    print(msg + "\n--- last 40 log lines ---")
    print("".join(open(LOG).readlines()[-40:]))
    raise SystemExit(msg)

deadline, shown, tick = time.time() + 45*60, 0, 0
while time.time() < deadline:
    if server.poll() is not None:
        die("vLLM exited during startup.")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models",
                        headers={"Authorization": f"Bearer {API_KEY}"}, timeout=5).ok:
            step("model loaded"); break
    except requests.RequestException:
        pass
    lines = open(LOG).readlines()
    for ln in lines[shown:]:
        if any(k in ln for k in ("Loading", "loading", "error", "Error",
                                 "CUDA", "out of memory", "%|", "KV cache")):
            print("   " + ln.rstrip()[:150], flush=True)
    shown = len(lines)
    tick += 1
    if tick % 4 == 0:
        step(f"still loading, {len(lines)} log lines so far")
    time.sleep(15)
else:
    die("timed out waiting for vLLM.")

step("opening tunnel")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(40):
    time.sleep(3)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",
                  open("/kaggle/working/tunnel.log").read())
    if m: PUBLIC = m.group(0); break
assert PUBLIC, "tunnel failed; see /kaggle/working/tunnel.log"

step("testing end to end through the tunnel")
ok = requests.post(PUBLIC + "/v1/chat/completions",
                   headers={"Authorization": f"Bearer {API_KEY}"},
                   json={"model": "qwen", "max_tokens": 8, "temperature": 0,
                         "messages": [{"role": "user", "content": "say: ready"}]},
                   timeout=180).ok

print("\n" + "="*70)
print("  ENDPOINT LIVE" if ok else "  TUNNEL UP, TEST CALL FAILED")
print("="*70)
print(f"  OPENAI_BASE_URL = {PUBLIC}/v1")
print(f"  OPENAI_API_KEY  = {API_KEY}")
print("  model name      = qwen")
print(f"  total setup time: {(time.time()-T0)/60:.1f} min")
print("="*70)
print("  Both change every session. Keep them in a .env, never in git.")

In [ ]:
# Skipped when BUILD_CACHE is True. Jupyter swallows SystemExit, so
# "Run All" would otherwise carry on and rebuild the environment.
if CACHE_DONE:
    raise RuntimeError("BUILD_CACHE run finished. Save Version, set "
                       "BUILD_CACHE=False, attach the output, then rerun.")

# 5. Hold the endpoint open, then stop the model automatically.
#    Interrupt this cell to stop sooner.

end = time.time() + RUN_MINUTES * 60
step(f"serving until {datetime.datetime.fromtimestamp(end):%H:%M:%S}")
try:
    while time.time() < end:
        if server.poll() is not None:
            print("vLLM died:"); print("".join(open(LOG).readlines()[-30:])); break
        step(f"alive, {int((end-time.time())/60)} min left  |  {PUBLIC}")
        time.sleep(300)
finally:
    for p in (tunnel, server):
        try: p.terminate()
        except Exception: pass
    time.sleep(5)
    print("\n" + "="*70)
    print("  Model stopped. GPU MEMORY is free.")
    print("")
    print("  YOUR QUOTA IS STILL RUNNING.")
    print("  Kaggle charges for an ATTACHED GPU session, not for GPU use.")
    print("  Click STOP SESSION in the sidebar. That is the only thing")
    print("  that stops the meter. The last cell is a fallback.")
    print("="*70)

## Saving the cache: exactly what to click

1. Set `BUILD_CACHE = True`, **Run All**. It builds `/kaggle/working/wheels` and `/kaggle/working/bin`, then stops deliberately.
2. Click **Save Version → Save & Run All (Commit)**. This is what publishes `/kaggle/working` as the notebook's **Output**. Interactive runs do **not** persist by themselves.
3. Set `BUILD_CACHE = False`.
4. **Add Input → Notebook Output →** pick this notebook. It mounts under `/kaggle/input/<notebook-slug>`. Set `CACHE` to that path.
5. **Stop Session.**

Note that step 2 re-runs the notebook in batch mode, which spends a little more quota. It is the only way Kaggle persists output, and you do it once.

## What actually stops the quota

| Action | Frees GPU memory | **Stops quota** |
|---|---|---|
| Timer expires, or you interrupt the cell | Yes | **No** |
| Closing the browser tab | No | **No** |
| **Stop Session** in the sidebar | Yes | **Yes** |
| Kaggle's own idle timeout | Yes | Yes, eventually |

Kaggle bills for a GPU session being **attached**, not for the GPU doing work.

## If the model download is slow, cache it too

The 20 GB `/kaggle/working` limit blocks saving a 19.5 GB model through Save Version. Private **Datasets** allow 200 GB, so use the API instead:

1. Get a token: kaggle.com → Settings → **Create New API Token**.
2. In the notebook: **Add-ons → Secrets**, add `KAGGLE_USERNAME` and `KAGGLE_KEY`.
3. Download the model to `/kaggle/tmp/model` (~60 GB free there), write a `dataset-metadata.json`, and run `kaggle datasets create -p /kaggle/tmp/model --dir-mode zip`.
4. Attach it and set `MODEL_CACHE` to its path.

Worth doing **only if** the measured download time justifies it. `hf-xet` plus 16 parallel workers is usually quick on Kaggle's network, and cell 2 prints the elapsed time so you can decide on evidence.

## Using it from the laptop

```python
from openai import OpenAI
client = OpenAI(base_url="<OPENAI_BASE_URL>", api_key="<OPENAI_API_KEY>")
print(client.chat.completions.create(
    model="qwen", messages=[{"role": "user", "content": "hello"}]
).choices[0].message.content)
```

## If it fails

| Symptom | Fix |
|---|---|
| CUDA out of memory | `MAX_LEN = 4096`. Still failing: try the cyankiwi model, or a smaller model on one T4 |
| Only 1 GPU visible | Accelerator is not **T4 x2** |
| `pip install --no-index` fails | Kaggle changed its base image, so cached wheels no longer match. Rebuild the cache |
| Download stuck at 0% | **Internet** is off in the sidebar |
| Marlin / FlashAttention error | Expected on Turing; already avoided by `awq` and `enforce-eager` |
| URL stopped working | Session ended. Run All again; URL and key are new |

## Honest caveats

- **Not yet run on real hardware.** Settings follow from documented T4 limits and the measured 19.5 GB weight size. The first run is the test.
- **~5.5 GB per GPU left for KV cache.** Tight. Fallback is a smaller model on one T4, which is fine here: the LLM only handles language, never computes a number, never decides an action.
- **Cached wheels are tied to Kaggle's Python and CUDA versions.** An image update can break the offline install and require a rebuild.

In [ ]:
# OPTIONAL FALLBACK: end the container from inside the notebook.
#
# Kills every process, terminating the session and stopping the quota.
# Blunt: anything unsaved in /kaggle/working is lost and the notebook will
# look like it crashed. Expected, not a failure.

CONFIRM = False   # set True, then run this cell

if CONFIRM:
    import signal
    print("terminating container...")
    os.kill(1, signal.SIGKILL)
else:
    print("Not armed. Set CONFIRM = True to terminate.")
    print("Prefer Stop Session in the sidebar.")